# Qwen no Colab: experimento manual

**Estado em 13 de outubro de 2026: o usuário relatou uma resposta curta em português concluída com Qwen3-8B no Colab. O agente não executou nem verificou independentemente a sessão. Sem saída tensors_cuda0 fornecida, a auditoria da colocação real dos tensores permanece sem comprovação. Isto não ativa IA no Tatu nem comprova o ramo 27B ou capacidade geral.**

Somente se uma cópia antiga falhar ao imprimir hf_device_map após carregar o modelo, use a [célula de recuperação](https://github.com/tatu-project/tatu/blob/main/docs/COLAB-EXPERIMENT.md) antes da pergunta pública, sem reinstalar nem recarregar pesos. Quem já obteve a resposta não precisa repetir a recuperação.

Abra uma cópia privada no **Colab hospedado**, selecione uma GPU em **Ambiente de execução → Alterar tipo de ambiente de execução** e execute as células em ordem. Comece com **Qwen3-8B**. Não conecte um runtime local.

Este experimento não atende usuários do Tatu, não cria servidor, túnel ou tarefa em segundo plano e não fornece disponibilidade contínua. A [FAQ oficial do Colab](https://research.google.com/colaboratory/faq.html) prioriza uso interativo; a GPU e a duração são variáveis, com até 12 horas no gratuito conforme disponibilidade e uso.

Use apenas o prompt público de demonstração. O texto é processado na infraestrutura do Colab/Google; saídas podem ser salvas no notebook. Não inclua senhas, chaves, conversas ou memórias privadas. Não há montagem do Drive nem autenticação no Hugging Face.

In [ ]:
# Altere apenas estes controles; reexecute o preflight após qualquer mudança.
MODEL_VARIANT = "8B"  # @param ["8B", "27B"]
ALLOW_27B_EXPERIMENT = False  # @param {type:"boolean"}

MODELS = {
    "8B": {
        "id": "Qwen/Qwen3-8B",
        "revision": "b968826d9c46dd6066d109eabc6255188de91218",
        "min_free_vram_gib": 10,
        "min_free_ram_gib": 8,
        "min_free_disk_gib": 24,
    },
    "27B": {
        "id": "Qwen/Qwen3.8-27B",
        "revision": "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0",
        "min_free_vram_gib": 24,
        "min_free_ram_gib": 20,
        "min_free_disk_gib": 80,
    },
}
PACKAGE_PINS = {
    "transformers": "5.19.0",
    "accelerate": "1.15.0",
    "bitsandbytes": "0.50.0",
}
MAX_INPUT_TOKENS = 256
MAX_NEW_TOKENS = 128
PREFLIGHT_OK = False
PACKAGES_OK = False


## Verifique antes de baixar

Os limites abaixo são **critérios conservadores deste experimento**, não mínimos oficiais nem garantia de funcionamento. “Livre” é a memória disponível agora na GPU 0, não a soma de GPUs.

O ramo **27B** é opcional e exige autorização explícita no controle acima e pelo menos **24 GiB livres de VRAM**, **20 GiB livres de RAM** e **80 GiB de disco**. Uma placa anunciada como “24 GB” pode não passar este teste. Uma T4 comum não passa. Mesmo passando, a arquitetura híbrida, estados, embeddings, kernels, quantização e cache podem causar falta de memória.

A quantização acontece ao carregar os arquivos originais: o download não encolhe para 4 bits. O repositório oficial contém aproximadamente 16,4 GB para 8B e 55,6 GB para 27B. Não há offload para CPU/disco nem troca automática por outro modelo.

In [ ]:
import csv
import importlib.metadata
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys

GIB = 1024 ** 3
CACHE_DIR = Path("/content/qwen-colab-experiment-cache")

def require_hosted_colab():
    if (
        platform.system() != "Linux"
        or not Path("/content").is_dir()
        or not os.environ.get("COLAB_RELEASE_TAG")
    ):
        raise RuntimeError("Use um runtime hospedado do Colab; execução local recusada.")

def available_ram_gib():
    fields = {}
    for line in Path("/proc/meminfo").read_text().splitlines():
        name, value = line.split(":", 1)
        fields[name] = int(value.strip().split()[0]) * 1024
    available = fields["MemAvailable"]
    # Considere também o limite do contêiner quando o Colab o expuser.
    for limit_path, used_path in [
        ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
        ("/sys/fs/cgroup/memory/memory.limit_in_bytes",
         "/sys/fs/cgroup/memory/memory.usage_in_bytes"),
    ]:
        limit_file, used_file = Path(limit_path), Path(used_path)
        if limit_file.is_file() and used_file.is_file():
            limit_text = limit_file.read_text().strip()
            if limit_text.isdigit():
                limit = int(limit_text)
                if limit < 2 ** 60:
                    available = min(available, max(0, limit - int(used_file.read_text())))
    return available / GIB

def run_preflight():
    global PREFLIGHT_OK, PREFLIGHT_VARIANT, TORCH_VERSION
    PREFLIGHT_OK = False
    require_hosted_colab()
    if MODEL_VARIANT not in MODELS:
        raise RuntimeError("Escolha 8B ou 27B.")
    if MODEL_VARIANT == "27B" and ALLOW_27B_EXPERIMENT is not True:
        raise RuntimeError("27B desativado. Confirme o experimento explicitamente.")
    report = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
         "--format=csv,noheader,nounits"],
        check=True, capture_output=True, text=True, timeout=10,
    )
    print("GPU(s), VRAM total e livre em MiB:\n" + report.stdout.strip())
    if not list(csv.reader(report.stdout.splitlines())):
        raise RuntimeError("Nenhuma GPU NVIDIA disponível.")
    import torch  # PyTorch fornecido pelo Colab; não é instalado por este notebook.
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA indisponível. Nenhum fallback para CPU será usado.")
    version = tuple(int(part) for part in torch.__version__.split("+")[0].split(".")[:2])
    if version < (2, 5):
        raise RuntimeError("O PyTorch do runtime precisa ser >=2.5. Use outro runtime.")
    cuda_version = tuple(int(part) for part in torch.version.cuda.split(".")[:2])
    if not (11, 8) <= cuda_version <= (13, 0):
        raise RuntimeError("CUDA fora da faixa documentada para os pacotes fixados.")
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    measured = {
        "free_vram_gib": free_bytes / GIB,
        "total_vram_gib": total_bytes / GIB,
        "free_ram_gib": available_ram_gib(),
        "free_disk_gib": shutil.disk_usage("/content").free / GIB,
    }
    settings = MODELS[MODEL_VARIANT]
    print({key: round(value, 2) for key, value in measured.items()})
    print("Modelo:", settings["id"], "revisão:", settings["revision"])
    for resource in ("vram", "ram", "disk"):
        required = settings["min_free_" + resource + "_gib"]
        if measured["free_" + resource + "_gib"] < required:
            raise RuntimeError(
                f"{resource.upper()} livre insuficiente: exige {required} GiB. "
                "Nenhum peso foi baixado por esta etapa."
            )
    TORCH_VERSION = importlib.metadata.version("torch")
    print({"python": sys.version.split()[0], "torch": TORCH_VERSION, "cuda": torch.version.cuda})
    PREFLIGHT_VARIANT = MODEL_VARIANT
    PREFLIGHT_OK = True
    return measured

PREFLIGHT_REPORT = run_preflight()


## Pacotes somente na VM do Colab

As três versões diretas são fixadas. PyTorch/CUDA vêm do runtime e são registrados; um arquivo de constraints impede que o instalador substitua silenciosamente o PyTorch. As dependências transitivas e a imagem do Colab podem variar, portanto compatibilidade real ainda precisa ser testada.

Esta célula não instala nada no repositório ou no computador. Se a instalação falhar, interrompa: não relaxe os limites, não ative código remoto nem adicione kernels de procedência desconhecida.

In [ ]:
assert PREFLIGHT_OK and PREFLIGHT_VARIANT == MODEL_VARIANT, "Execute o preflight."
require_hosted_colab()
PACKAGES_OK = False
CACHE_DIR.mkdir(parents=True, exist_ok=True)
constraint = CACHE_DIR / "torch-constraint.txt"
constraint.write_text(f"torch=={TORCH_VERSION}\n")
pip_environment = dict(os.environ)
pip_environment.update({
    "PIP_CONFIG_FILE": "/dev/null",
    "PIP_EXTRA_INDEX_URL": "",
    "PIP_NO_INPUT": "1",
})
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--only-binary=:all:",
     "--index-url", "https://pypi.org/simple", "--constraint", str(constraint)]
    + [f"{name}=={version}" for name, version in PACKAGE_PINS.items()],
    check=True, env=pip_environment,
)
for name, expected in PACKAGE_PINS.items():
    actual = importlib.metadata.version(name)
    if actual != expected:
        raise RuntimeError(f"Versão inesperada de {name}; reinicie o runtime.")
    print(name, actual)
if importlib.metadata.version("torch") != TORCH_VERSION:
    raise RuntimeError("PyTorch foi alterado; interrompa e reinicie o runtime.")
PACKAGES_OK = True


## Carregar uma vez, apenas na GPU

8B usa a classe nativa causal; 27B usa a classe nativa `Qwen3_5ForConditionalGeneration` declarada no config oficial. O experimento 27B é apenas de texto, embora carregue um checkpoint multimodal. Não há visão, áudio nem kernels opcionais neste teste.

`trust_remote_code=False`, pesos Safetensors, revisão fixada, token desativado e mapa explícito para GPU 0. A implementação PyTorch de atenção híbrida no 27B pode ser lenta e consumir mais memória. Se ocorrer OOM, execute a liberação e encerre/reinicie a VM; não há tentativa automática na CPU.

In [ ]:
def require_model_cuda0(model):
    parameters = 0
    tensors = 0
    for kind, values in (("parameter", model.parameters()), ("buffer", model.buffers())):
        for tensor in values:
            device = tensor.device
            if device.type != "cuda" or device.index != 0:
                raise RuntimeError("Offload recusado: todos os tensores devem estar em cuda:0.")
            if kind == "parameter":
                parameters += 1
            tensors += 1
    if parameters == 0:
        raise RuntimeError("Modelo vazio: nenhum parametro encontrado.")
    return tensors

def optional_device_map(model):
    device_map = getattr(model, "hf_device_map", None)
    if not isinstance(device_map, dict):
        return None
    return {key: str(value) for key, value in device_map.items()}

assert PREFLIGHT_OK and PREFLIGHT_VARIANT == MODEL_VARIANT, "Execute o preflight."
assert PACKAGES_OK, "Instale os pacotes na célula anterior."
require_hosted_colab()
if globals().get("model") is not None:
    raise RuntimeError("Libere o modelo anterior antes de carregar novamente.")
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HOME"] = str(CACHE_DIR / "huggingface")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
if MODEL_VARIANT == "27B":
    from transformers import Qwen3_5ForConditionalGeneration
    loader = Qwen3_5ForConditionalGeneration
else:
    loader = AutoModelForCausalLM

# Repita os limites antes de qualquer download de tokenizer/config/pesos.
run_preflight()
settings = MODELS[MODEL_VARIANT]
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
tokenizer = AutoTokenizer.from_pretrained(
    settings["id"], revision=settings["revision"],
    cache_dir=str(CACHE_DIR / "weights"), token=False, trust_remote_code=False,
)
model = None
torch.cuda.reset_peak_memory_stats(0)
try:
    model = loader.from_pretrained(
        settings["id"], revision=settings["revision"],
        cache_dir=str(CACHE_DIR / "weights"), token=False,
        trust_remote_code=False, use_safetensors=True,
        quantization_config=quantization, dtype=compute_dtype,
        device_map={"": 0}, low_cpu_mem_usage=True, attn_implementation="sdpa",
    )
    require_model_cuda0(model)
    model.eval()
except Exception:
    model = None
    gc.collect()
    torch.cuda.empty_cache()
    print("Carga não concluída. Libere os recursos e reinicie a VM se necessário.")
    raise

LOAD_PEAK_GIB = torch.cuda.max_memory_allocated(0) / GIB
print({
    "model_footprint_gib": round(model.get_memory_footprint() / GIB, 2),
    "load_peak_allocated_gib": round(LOAD_PEAK_GIB, 2),
    "free_vram_gib": round(torch.cuda.mem_get_info(0)[0] / GIB, 2),
    "dtype": str(compute_dtype),
    "device_map": optional_device_map(model),
})


## Uma pergunta pública, curta e sem ferramentas

Pensamento desativado no template, entrada limitada a 256 tokens e saída limitada a 128 tokens. Não há histórico privado, consulta à internet ou conexão com o Tatu. A geração tem um limite suave de 60 segundos: uma operação já em execução pode terminar depois desse tempo.

In [ ]:
assert globals().get("model") is not None, "Carregue o modelo primeiro."
require_hosted_colab()
require_model_cuda0(model)
import time

PUBLIC_PROMPT = (
    "Em português brasileiro, explique em até três frases o que faz "
    "um assistente pessoal de IA. Não invente funções já implementadas no Tatu."
)
messages = [{"role": "user", "content": PUBLIC_PROMPT}]
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
)
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
input_tokens = int(inputs["input_ids"].shape[-1])
if input_tokens > MAX_INPUT_TOKENS:
    raise RuntimeError("Entrada grande demais; reduza o prompt sem truncamento oculto.")
inputs = {key: value.to("cuda:0") for key, value in inputs.items()}
torch.manual_seed(42)
torch.cuda.reset_peak_memory_stats(0)
torch.cuda.synchronize(0)
started = time.perf_counter()
with torch.inference_mode():
    generated = model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS, max_time=60,
        do_sample=True, temperature=0.7, top_p=0.8, top_k=20,
        pad_token_id=tokenizer.eos_token_id,
    )
torch.cuda.synchronize(0)
elapsed = time.perf_counter() - started
reply_ids = generated[0, input_tokens:].detach().cpu()
print(tokenizer.decode(reply_ids, skip_special_tokens=True))
print({
    "model": settings["id"],
    "revision": settings["revision"],
    "input_tokens": input_tokens,
    "output_tokens": int(reply_ids.shape[-1]),
    "seconds": round(elapsed, 2),
    "peak_allocated_gib": round(torch.cuda.max_memory_allocated(0) / GIB, 2),
    "reserved_gib": round(torch.cuda.memory_reserved(0) / GIB, 2),
    "free_vram_gib": round(torch.cuda.mem_get_info(0)[0] / GIB, 2),
})


## Liberar e encerrar

Execute a última célula também depois de um erro. Ela remove referências e libera o cache CUDA; isso não promete encerrar a VM nem remover o contexto do driver. Depois use **Ambiente de execução → Desconectar e excluir ambiente de execução**. Os pesos no disco efêmero desaparecem quando a VM é excluída.

Limpe as saídas antes de compartilhar a cópia. A execução deste notebook não ativa a IA de produção do Tatu.

In [ ]:
# Não requer download, instalação ou servidor.
import gc
for name in (
    "model", "tokenizer", "inputs", "generated", "reply_ids",
    "messages", "text", "quantization", "loader",
):
    globals().pop(name, None)
gc.collect()
if "torch" in globals() and torch.cuda.is_available():
    torch.cuda.empty_cache()
    print({
        "allocated_after_cleanup_gib": round(torch.cuda.memory_allocated(0) / (1024 ** 3), 3),
        "free_vram_after_cleanup_gib": round(torch.cuda.mem_get_info(0)[0] / (1024 ** 3), 2),
    })
PREFLIGHT_OK = False
PACKAGES_OK = False
print("Objetos liberados. Agora desconecte e exclua o runtime pelo menu do Colab.")


## Fontes oficiais e estado de verificação

Consultadas em **13/10/2026**:

- [Qwen3-8B, revisão fixada](https://huggingface.co/Qwen/Qwen3-8B/tree/b968826d9c46dd6066d109eabc6255188de91218).
- [Qwen3.8-27B, revisão fixada](https://huggingface.co/Qwen/Qwen3.8-27B/tree/1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0).
- [Transformers 5.19.0](https://pypi.org/project/transformers/5.19.0/), [Accelerate 1.15.0](https://pypi.org/project/accelerate/1.15.0/), [bitsandbytes 0.50.0](https://pypi.org/project/bitsandbytes/0.50.0/).
- [Quantização bitsandbytes](https://huggingface.co/docs/transformers/v5.19.0/quantization/bitsandbytes), [classes Qwen3.5](https://huggingface.co/docs/transformers/v5.19.0/model_doc/qwen3_5), [FAQ Colab](https://research.google.com/colaboratory/faq.html).

Os pesos têm licença Apache-2.0, independentemente da licença MIT do Tatu. A verificação deste artefato é estática. Disponibilidade de GPU, instalação na VM, carregamento, consumo real e resposta só podem ser confirmados após uma execução manual no Colab.